In [38]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
STATE_DIM = 503
N_ACTIONS = 7  


N_EPISODES_TRAIN = 90
N_EPISODES_VAL = 6

torch.manual_seed(SEED)
np.random.seed(SEED)


class BCDataset(Dataset):
    #custom dataset
    def __init__(self, df: pd.DataFrame, episode_ids, state_cols):
        sub = df[df["episode_id"].isin(episode_ids)]
        self.X = sub[state_cols].to_numpy(dtype=np.float32).copy()
        self.y = sub["action"].to_numpy(dtype=np.int64).copy()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), self.y[idx]


def split_episodes(episode_ids, n_train=N_EPISODES_TRAIN, n_val=N_EPISODES_VAL, seed=SEED):
    rng = np.random.default_rng(seed)  # random num generator
    ids = np.array(sorted(episode_ids)) 
    rng.shuffle(ids)

    n_total = len(ids)
    if n_train + n_val >= n_total:
        raise ValueError(
            f"Not enough episodes ({n_total}) for N_EPISODES_TRAIN={n_train} "
        )

    train_ids = ids[:n_train]
    val_ids = ids[n_train:n_train + n_val]
    test_ids = ids[n_train + n_val:]
    return train_ids, val_ids, test_ids


def load_data(csv_path, batch_size=256, num_workers=0):
    df = pd.read_csv(csv_path)

    if "episode_id" not in df.columns:
        raise ValueError(
            "Expected an 'episode_id' column"
        )

    state_cols = [f"s{i}" for i in range(STATE_DIM)]
    missing = [c for c in state_cols + ["action"] if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing expected columns: {missing[:10]}...")

    before = len(df)
    #remove missing columns
    df = df[pd.to_numeric(df["action"], errors="coerce").isin(range(N_ACTIONS))].copy() 
    df["action"] = df["action"].astype(int)
    dropped = before - len(df)
    if dropped:
        print(f"Warning: dropped {dropped} rows with invalid action values (corrupted rows)")

    episode_ids = df["episode_id"].unique()
    train_ids, val_ids, test_ids = split_episodes(episode_ids)
    print(f"Episodes -> train: {len(train_ids)}, val: {len(val_ids)}, test: {len(test_ids)}")

    train_ds = BCDataset(df, train_ids, state_cols)
    val_ds = BCDataset(df, val_ids, state_cols)
    test_ds = BCDataset(df, test_ids, state_cols)
    print(f"Rows -> train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    counts = np.bincount(train_ds.y, minlength=N_ACTIONS).astype(np.float32)
    counts[counts == 0] = 1.0  # preventing div by 0 error provided an action never occurs
    raw_weights = counts.sum() / (N_ACTIONS * counts) # compensate for rare actions
    class_weights = np.sqrt(raw_weights)
    class_weights = np.clip(class_weights, None, 10.0)  # cap max weight ratio
    print("Action counts (train):", counts.astype(int).tolist())
    print("Class weights (sqrt-dampened, capped at 10x):", np.round(class_weights, 3).tolist())

    return train_loader, val_loader, test_loader, torch.tensor(class_weights)


class BCPolicy(nn.Module):

    def __init__(self, input_dim=STATE_DIM, hidden=(256, 128), n_actions=N_ACTIONS, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_actions))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # logits

def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    if train:
        model.train()
    else:
        model.eval()
    total_loss, correct, total = 0.0, 0, 0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            if train:
                optimizer.zero_grad()

            logits = model(X)
            loss = criterion(logits, y)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += X.size(0)

    return total_loss / total, correct / total


def train(csv_path, epochs=30, batch_size=256, lr=1e-3, patience=5, out_path="bc_policy.pt"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_loader, val_loader, test_loader, class_weights = load_data(csv_path, batch_size)

    model = BCPolicy().to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    # criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)

        print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.3f} "
              f"| val loss {val_loss:.4f} acc {val_acc:.3f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), out_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no val improvement for {patience} epochs)")
                break

    # evaluate best checkpoint on held-out test episodes
    model.load_state_dict(torch.load(out_path))
    test_loss, test_acc = run_epoch(model, test_loader, criterion, optimizer, device, train=False)
    print(f"\nTest (held-out episodes) | loss {test_loss:.4f} acc {test_acc:.3f}")
    print(f"Best model saved to {out_path}")

    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X, y in test_loader:
            preds = model(X.to(device)).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_true.append(y)
    all_preds = torch.cat(all_preds).numpy()
    all_true = torch.cat(all_true).numpy()

    from sklearn.metrics import classification_report, confusion_matrix
    print(classification_report(all_true, all_preds, labels=list(range(N_ACTIONS)), zero_division=0))
 
    print("Confusion matrix (rows=true, cols=predicted, actions 0-6):")
    cm = confusion_matrix(all_true, all_preds, labels=list(range(N_ACTIONS)))
    print(cm)


    return model

In [39]:
model = train("../dataset_full.csv", epochs=30, batch_size=256, lr=1e-3, patience=5, out_path="bc_policy.pt")

Device: cuda
Episodes -> train: 90, val: 6, test: 4
Rows -> train: 55449, val: 2952, test: 2794
Action counts (train): [12890, 14020, 230, 353, 18068, 1799, 8089]
Class weights (sqrt-dampened, capped at 10x): [0.7839999794960022, 0.7519999742507935, 5.86899995803833, 4.736999988555908, 0.6620000004768372, 2.0980000495910645, 0.9900000095367432]
Epoch 01 | train loss 0.8267 acc 0.789 | val loss 0.4612 acc 0.864
Epoch 02 | train loss 0.5251 acc 0.825 | val loss 0.4056 acc 0.872
Epoch 03 | train loss 0.4795 acc 0.834 | val loss 0.3809 acc 0.873
Epoch 04 | train loss 0.4579 acc 0.838 | val loss 0.3977 acc 0.875
Epoch 05 | train loss 0.4449 acc 0.843 | val loss 0.3914 acc 0.879
Epoch 06 | train loss 0.4332 acc 0.845 | val loss 0.3802 acc 0.884
Epoch 07 | train loss 0.4258 acc 0.847 | val loss 0.3894 acc 0.875
Epoch 08 | train loss 0.4214 acc 0.850 | val loss 0.3859 acc 0.882
Epoch 09 | train loss 0.4139 acc 0.852 | val loss 0.3955 acc 0.887
Epoch 10 | train loss 0.4088 acc 0.853 | val loss 